# 15 — dbt og Snowflake for AI-team

**Fase:** 3 — Dataengineering | **Tid:** 2 timer | **Krav:** Notatbok 14

**Hva du bygger:** Et lite dbt-prosjekt som transformerer pensjonsdata med testbare SQL-modeller — kjørt lokalt med DuckDB (ingen Snowflake-konto nødvendig).

---

## Hva er dbt?

**dbt** (data build tool) er standarden for SQL-transformasjoner i moderne datastack.

```
Uten dbt:  sql-fil → kjør → håp at det funker
Med dbt:   modell → test → dokumentasjon → lineage-graf
```

dbt tilfører tre ting til SQL:
1. **Modeller** — SQL-filer som kompileres og kjøres i riktig rekkefølge
2. **Tester** — Automatiske datavalidering (ikke null, unik, gyldige verdier)
3. **Dokumentasjon** — Automatisk generert fra koden

## Kobling til Snowflake

I SPK-annonsen nevnes **dbt + Snowflake** som dataplattform. De fungerer identisk:  
`dbt-core` med `dbt-duckdb` lokalt = samme konsepter som `dbt-snowflake` i sky.

In [ ]:
%pip install -q dbt-core dbt-duckdb duckdb

---

## Del 1: Sett opp dbt-prosjekt

In [ ]:
import os
import pathlib

# Lag dbt-prosjektstruktur
PROSJEKT = pathlib.Path("spk_dbt")

mapper = [
    PROSJEKT / "models" / "staging",
    PROSJEKT / "models" / "mart",
    PROSJEKT / "seeds",
    PROSJEKT / "tests",
]
for m in mapper:
    m.mkdir(parents=True, exist_ok=True)

print("Mappestruktur:")
for m in sorted(PROSJEKT.rglob("*")):
    print(f"  {m}")

In [ ]:
# dbt_project.yml — konfigurasjonsfil
(PROSJEKT / "dbt_project.yml").write_text("""\
name: 'spk_dbt'
version: '1.0.0'
profile: 'spk_dbt'
model-paths: ['models']
seed-paths: ['seeds']
test-paths: ['tests']

models:
  spk_dbt:
    staging:
      +materialized: view
    mart:
      +materialized: table
""")

# profiles.yml — databasetilkobling (DuckDB lokalt)
profiles_dir = pathlib.Path.home() / ".dbt"
profiles_dir.mkdir(exist_ok=True)
(profiles_dir / "profiles.yml").write_text("""\
spk_dbt:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: '../spk_warehouse.duckdb'
""")

print("Konfigurasjon skrevet.")

---

## Del 2: Seed — Last inn kildedata

In [ ]:
# Seed-fil: rådata som CSV som dbt kan laste inn
(PROSJEKT / "seeds" / "raw_members.csv").write_text("""\
member_id,navn,aarslonn,opptjeningsaar,pensjonstype,aktiv
1,Kari Nordmann,650000,28,AFP,true
2,Ola Hansen,580000,35,Alderspensjon,true
3,Anna Berg,720000,15,AFP,true
4,Per Dahl,490000,40,Alderspensjon,false
5,Eva Lund,620000,22,Uforepensjon,true
6,Lars Moe,,10,AFP,true
7,Knut Vik,450000,32,Alderspensjon,true
""")
print("Seed-fil klar.")

---

## Del 3: Modeller

In [ ]:
# Staging-modell: Rens og standardiser rådata
(PROSJEKT / "models" / "staging" / "stg_members.sql").write_text("""\
-- Staging-modell: renser og standardiserer rådata
-- {{ ref('raw_members') }} er dbt-syntaks for å referere til en seed/modell

SELECT
    member_id,
    UPPER(navn)                          AS navn,
    COALESCE(aarslonn, 0)                AS aarslonn,     -- Erstatt NULL med 0
    opptjeningsaar,
    LOWER(pensjonstype)                  AS pensjonstype,
    aktiv,
    CURRENT_TIMESTAMP                    AS oppdatert_kl
FROM {{ ref('raw_members') }}
WHERE member_id IS NOT NULL
""")

# Mart-modell: Beregn pensjon (forretningslogikk)
(PROSJEKT / "models" / "mart" / "pensjon_beregning.sql").write_text("""\
-- Mart-modell: forretningslogikk for pensjonsberegning
-- Brukes av dashboards og AI-systemer

SELECT
    member_id,
    navn,
    aarslonn,
    opptjeningsaar,
    pensjonstype,

    -- Beregn opptjeningssats (maks 30 år = 100%)
    LEAST(opptjeningsaar / 30.0, 1.0)   AS opptjeningssats,

    -- Estimert månedlig pensjon
    ROUND(
        aarslonn * 0.66 * LEAST(opptjeningsaar / 30.0, 1.0) / 12
    , 0)                                 AS est_maanedlig_pensjon,

    -- Kategoriser
    CASE
        WHEN opptjeningsaar >= 30 THEN 'full_opptjening'
        WHEN opptjeningsaar >= 15 THEN 'delvis_opptjening'
        ELSE 'tidlig_fase'
    END                                  AS opptjeningskategori

FROM {{ ref('stg_members') }}
WHERE aktiv = true
""")

print("Modeller skrevet.")

---

## Del 4: Tests

In [ ]:
# schema.yml — definer tester for modellene
(PROSJEKT / "models" / "staging" / "schema.yml").write_text("""\
version: 2

models:
  - name: stg_members
    description: "Renset og standardisert medlemsdata"
    columns:
      - name: member_id
        description: "Unik ID for hvert medlem"
        tests:
          - unique
          - not_null
      - name: pensjonstype
        tests:
          - accepted_values:
              values: ['afp', 'alderspensjon', 'uforepensjon']
      - name: aarslonn
        tests:
          - not_null
""")

print("Test-konfigurasjon skrevet.")

In [ ]:
import subprocess

def kjør_dbt(kommando: str):
    """Kjør en dbt-kommando og vis output."""
    res = subprocess.run(
        f"cd {PROSJEKT} && dbt {kommando}",
        shell=True, capture_output=True, text=True
    )
    output = res.stdout + res.stderr
    # Vis kun de viktige linjene
    for linje in output.split("\n"):
        if any(k in linje for k in ["OK", "ERROR", "WARN", "PASS", "FAIL", "Completed", "of"]):
            print(linje)

print("Kjør dbt seed (last inn CSV):")
kjør_dbt("seed")

print("\nKjør dbt run (bygg modeller):")
kjør_dbt("run")

print("\nKjør dbt test (valider data):")
kjør_dbt("test")

In [ ]:
import duckdb

# Les resultater fra den ferdige dbt-modellen
con = duckdb.connect("spk_warehouse.duckdb")
df = con.execute("SELECT * FROM pensjon_beregning ORDER BY est_maanedlig_pensjon DESC").df()
print(df.to_string(index=False))

---

## Snowflake-kobling (konseptuelt)

```yaml
# profiles.yml for Snowflake (erstatt DuckDB-seksjonen)
spk_dbt:
  target: prod
  outputs:
    prod:
      type: snowflake
      account: xyz12345.eu-west-1
      user: dbt_user
      password: "{{ env_var('SNOWFLAKE_PASSWORD') }}"
      role: TRANSFORMER
      database: SPK_DW
      warehouse: COMPUTE_WH
      schema: marts
```

**Alt annet er identisk** — samme modeller, samme tester, samme kommandoer.

---

## Oppsummering

| dbt-konsept | Hva det er |
|-------------|----------|
| Modell | SQL-fil som kompileres til view/tabell |
| `{{ ref() }}` | Referanse mellom modeller (bygger lineage) |
| Seed | CSV-fil lastet inn som tabell |
| Test | Automatisk datavalidering |
| `dbt run` | Kjør alle modeller |
| `dbt test` | Kjør alle tester |

---

## Hva er neste steg?

**Neste: `16_prefect_orchestration.ipynb`** — Automatiser hele pipelinen med Prefect: schedules, retries, logging og betingede steg.